# Medical reward score playground

This notebook builds a single reward score for RL training where **ground truth = N** and **model output = M**.

It does four things:

1. Computes an order-invariant semantic score with **one-to-one matching**.
2. Converts that into a single **0 to 1** score with **F\(_\beta\)**.
3. Applies a **clinical penalty layer** for dangerous near-matches such as negation, laterality, dose, route, and frequency mismatches.
4. Lets you swap the local lexical backend for a real **medical embedding model** later, while keeping the reward logic unchanged.


## Reward shape

The core shape is:

\[
\text{reward} = \mathrm{clip}\left(F_\beta - \lambda \cdot \overline{\text{pair penalty}},\, 0,\, 1\right)
\]

where:

- \(F_\beta\) comes from one-to-one matching between model sentences and ground-truth sentences.
- \(\overline{\text{pair penalty}}\) is the average penalty across matched pairs.
- \(\lambda\) controls how aggressively the rule layer suppresses unsafe near-matches.

This notebook uses a **lexical fallback backend** by default so it can run locally with common Python packages. For real training, swap in a clinical embedding model and keep the rest unchanged.


In [1]:
import re
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

def to_items(text_or_items):
    if isinstance(text_or_items, str):
        parts = [x.strip() for x in re.split(r"[\n;]+", text_or_items) if x.strip()]
        return parts
    return [str(x).strip() for x in text_or_items if str(x).strip()]

In [2]:
STOPWORDS = {
    "a","an","the","of","for","and","or","to","in","on","at","by","with","from","is","are","was","were",
    "be","being","been","has","have","had","this","that","these","those","as","it","its","into","than",
    "patient","pt","reports","report","noted","note","states","stated"
}

NEGATION_PATTERNS = [
    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bdenies\b", r"\bdenied\b",
    r"\bnegative for\b", r"\bfree of\b", r"\babsence of\b"
]

LATERALITY_PATTERNS = {
    "bilateral": [r"\bbilateral\b"],
    "left": [r"\bleft\b", r"\blt\b"],
    "right": [r"\bright\b", r"\brt\b"],
}

EXPERIENCER_PATTERNS = {
    "family": [r"\bfamily history\b", r"\bmother\b", r"\bfather\b", r"\bsister\b", r"\bbrother\b", r"\bparent\b"],
}

TEMPORALITY_PATTERNS = {
    "history": [r"\bhistory of\b", r"\bhx of\b", r"\bprior\b", r"\bprevious\b", r"\bremote\b"],
    "chronic": [r"\bchronic\b", r"\blongstanding\b"],
    "acute": [r"\bacute\b", r"\bnew onset\b"],
    "current": [r"\bcurrently\b", r"\btoday\b", r"\bactive\b", r"\bpresenting with\b"],
}

ROUTE_PATTERNS = {
    "po": [r"\bpo\b", r"\boral\b", r"\bby mouth\b"],
    "iv": [r"\biv\b", r"\bintravenous\b"],
    "im": [r"\bim\b", r"\bintramuscular\b"],
    "sc": [r"\bsc\b", r"\bsubcutaneous\b", r"\bsubcut\b"],
    "inh": [r"\binhal(?:ed|ation|er)?\b"]
}

FREQ_PATTERNS = {
    "daily": [r"\bdaily\b", r"\bonce daily\b", r"\bqd\b"],
    "bid": [r"\bbid\b", r"\btwice daily\b"],
    "tid": [r"\btid\b", r"\bthree times daily\b"],
    "qid": [r"\bqid\b", r"\bfour times daily\b"],
    "prn": [r"\bprn\b", r"\bas needed\b"],
}

DURATION_RE = re.compile(r"\bfor\s+(\d+)\s*(day|days|week|weeks|month|months)\b", re.I)
QH_RE = re.compile(r"\bq(\d{1,2})h\b", re.I)
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9\-/]+")

NUMBER_UNIT_RE = re.compile(
    r"\b(\d+(?:\.\d+)?)\s*(mg/dl|mmol/l|mm hg|mmhg|mcg|μg|ug|mg|g|kg|ml|l|%|bpm)\b",
    re.I,
)

BP_RE = re.compile(r"\b(\d{2,3})\s*/\s*(\d{2,3})\s*mm\s*hg\b", re.I)

DEFAULT_PENALTY_WEIGHTS = {
    "negation": 1.0,
    "laterality": 1.0,
    "experiencer": 1.0,
    "temporality": 0.8,
    "numbers_units": 1.2,
    "route": 1.0,
    "frequency": 0.8,
    "duration": 0.6,
    "protected_terms": 1.0,
}

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())

def detect_first_category(text, patterns_dict):
    t = normalize_text(text)
    for label, patterns in patterns_dict.items():
        if any(re.search(p, t, re.I) for p in patterns):
            return label
    return None

def detect_temporality(text):
    t = normalize_text(text)
    tags = set()
    for label, patterns in TEMPORALITY_PATTERNS.items():
        if any(re.search(p, t, re.I) for p in patterns):
            tags.add(label)
    return tags

def normalize_number_unit(value, unit):
    unit = unit.lower().replace(" ", "")
    if unit in {"μg", "ug"}:
        unit = "mcg"
    if unit in {"mg", "g", "mcg"}:
        factor = {"mcg": 0.001, "mg": 1.0, "g": 1000.0}[unit]
        return ("mass", round(float(value) * factor, 6), "mg")
    if unit in {"ml", "l"}:
        factor = {"ml": 1.0, "l": 1000.0}[unit]
        return ("volume", round(float(value) * factor, 6), "ml")
    if unit == "kg":
        return ("weight", round(float(value), 6), "kg")
    if unit == "mmhg":
        return ("pressure", round(float(value), 6), "mmhg")
    if unit == "%":
        return ("percent", round(float(value), 6), "%")
    if unit == "bpm":
        return ("rate", round(float(value), 6), "bpm")
    if unit in {"mg/dl", "mmol/l"}:
        return ("lab", round(float(value), 6), unit)
    return ("other", round(float(value), 6), unit)

def extract_numbers_units(text):
    t = normalize_text(text)
    out = []
    for s, d in BP_RE.findall(t):
        out.append(("bp_sys", float(s), "mmhg"))
        out.append(("bp_dia", float(d), "mmhg"))
    for val, unit in NUMBER_UNIT_RE.findall(t):
        out.append(normalize_number_unit(val, unit))
    return sorted(set(out))

def detect_freq(text):
    t = normalize_text(text)
    for label, patterns in FREQ_PATTERNS.items():
        if any(re.search(p, t, re.I) for p in patterns):
            return label
    m = QH_RE.search(t)
    return f"q{m.group(1)}h" if m else None

def detect_duration(text):
    t = normalize_text(text)
    m = DURATION_RE.search(t)
    return f"{m.group(1)} {m.group(2)}" if m else None

def content_tokens(text):
    toks = [tok.lower() for tok in TOKEN_RE.findall(text.lower())]
    return set(t for t in toks if t not in STOPWORDS and not t.isdigit())

def extract_attrs(text):
    t = normalize_text(text)
    return {
        "negated": any(re.search(p, t, re.I) for p in NEGATION_PATTERNS),
        "laterality": detect_first_category(t, LATERALITY_PATTERNS),
        "experiencer": detect_first_category(t, EXPERIENCER_PATTERNS) or "patient",
        "temporality": detect_temporality(t),
        "route": detect_first_category(t, ROUTE_PATTERNS),
        "frequency": detect_freq(t),
        "duration": detect_duration(t),
        "numbers_units": extract_numbers_units(t),
        "content_tokens": sorted(content_tokens(t)),
    }

In [3]:
def lexical_similarity_matrix(model_items, gt_items):
    model_items = to_items(model_items)
    gt_items = to_items(gt_items)
    if len(model_items) == 0 or len(gt_items) == 0:
        return np.zeros((len(model_items), len(gt_items)))
    texts = [normalize_text(x) for x in model_items + gt_items]
    vect = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
    X = vect.fit_transform(texts)
    Xm = X[:len(model_items)]
    Xg = X[len(model_items):]
    return cosine_similarity(Xm, Xg)

def similarity_from_embeddings(model_embs, gt_embs):
    model_embs = np.asarray(model_embs, dtype=np.float32)
    gt_embs = np.asarray(gt_embs, dtype=np.float32)
    return model_embs @ gt_embs.T

def hungarian_match(sim_matrix, threshold=0.35):
    M, N = sim_matrix.shape
    if M == 0 or N == 0:
        return []
    rows, cols = linear_sum_assignment(1.0 - sim_matrix)
    pairs = []
    for i, j in zip(rows, cols):
        score = float(sim_matrix[i, j])
        if score >= threshold:
            pairs.append((int(i), int(j), score))
    return pairs

In [4]:
def compare_number_sets(nums1, nums2):
    return set(nums1) == set(nums2)

def pair_penalties(model_text, gt_text, weights=None, protected_terms=None):
    weights = {**DEFAULT_PENALTY_WEIGHTS, **(weights or {})}
    a = extract_attrs(model_text)
    b = extract_attrs(gt_text)
    penalties = {}

    if a["negated"] != b["negated"]:
        penalties["negation"] = weights["negation"]
    if a["laterality"] and b["laterality"] and a["laterality"] != b["laterality"]:
        penalties["laterality"] = weights["laterality"]
    if a["experiencer"] != b["experiencer"]:
        penalties["experiencer"] = weights["experiencer"]
    if a["temporality"] and b["temporality"] and a["temporality"] != b["temporality"]:
        penalties["temporality"] = weights["temporality"]
    if (a["numbers_units"] or b["numbers_units"]) and not compare_number_sets(a["numbers_units"], b["numbers_units"]):
        penalties["numbers_units"] = weights["numbers_units"]
    if a["route"] and b["route"] and a["route"] != b["route"]:
        penalties["route"] = weights["route"]
    if a["frequency"] and b["frequency"] and a["frequency"] != b["frequency"]:
        penalties["frequency"] = weights["frequency"]
    if a["duration"] and b["duration"] and a["duration"] != b["duration"]:
        penalties["duration"] = weights["duration"]

    if protected_terms:
        pt = {str(x).lower() for x in protected_terms}
        a_terms = set(a["content_tokens"]) & pt
        b_terms = set(b["content_tokens"]) & pt
        if (a_terms or b_terms) and a_terms != b_terms:
            penalties["protected_terms"] = weights["protected_terms"]

    return penalties

def fbeta_from_matched_total(matched_total, m_count, n_count, beta=1.0):
    if m_count == 0 and n_count == 0:
        return 1.0, 1.0, 1.0
    if m_count == 0 or n_count == 0:
        return 0.0, 0.0, 0.0
    precision = matched_total / m_count
    recall = matched_total / n_count
    if precision == 0 and recall == 0:
        return 0.0, precision, recall
    fbeta = (1 + beta**2) * precision * recall / (beta**2 * precision + recall)
    return float(fbeta), float(precision), float(recall)

def reward_from_similarity(
    model_items,
    gt_items,
    sim_matrix,
    beta=1.0,
    sim_threshold=0.35,
    penalty_weights=None,
    penalty_scale=0.75,
    protected_terms=None,
):
    model_items = to_items(model_items)
    gt_items = to_items(gt_items)
    sim_matrix = np.asarray(sim_matrix)
    assert sim_matrix.shape == (len(model_items), len(gt_items))
    pairs = hungarian_match(sim_matrix, threshold=sim_threshold)
    matched_total = sum(score for _, _, score in pairs)
    fbeta, precision, recall = fbeta_from_matched_total(
        matched_total, len(model_items), len(gt_items), beta=beta
    )

    matched_model = set()
    matched_gt = set()
    pair_rows = []
    total_penalty = 0.0

    for i, j, score in pairs:
        matched_model.add(i)
        matched_gt.add(j)
        penalties = pair_penalties(
            model_items[i],
            gt_items[j],
            weights=penalty_weights,
            protected_terms=protected_terms,
        )
        pair_penalty = float(sum(penalties.values()))
        total_penalty += pair_penalty
        pair_rows.append(
            {
                "model_index": i,
                "gt_index": j,
                "similarity": round(score, 4),
                "model_text": model_items[i],
                "gt_text": gt_items[j],
                "penalties": ", ".join(f"{k}:{v}" for k, v in penalties.items()) if penalties else "",
                "pair_penalty": round(pair_penalty, 4),
            }
        )

    avg_pair_penalty = total_penalty / max(len(pairs), 1)
    reward = float(np.clip(fbeta - penalty_scale * avg_pair_penalty, 0.0, 1.0))

    return {
        "reward": round(reward, 4),
        "semantic_fbeta": round(fbeta, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "matched_total": round(matched_total, 4),
        "avg_pair_penalty": round(avg_pair_penalty, 4),
        "pairs": pair_rows,
        "pairs_df": pd.DataFrame(pair_rows),
        "unmatched_model": [model_items[i] for i in range(len(model_items)) if i not in matched_model],
        "unmatched_ground_truth": [gt_items[j] for j in range(len(gt_items)) if j not in matched_gt],
        "sim_matrix": sim_matrix,
    }

def reward_score_lexical(model_items, gt_items, **kwargs):
    sim = lexical_similarity_matrix(model_items, gt_items)
    return reward_from_similarity(model_items, gt_items, sim, **kwargs)

def reward_score_from_embeddings(model_items, gt_items, model_embs, gt_embs, **kwargs):
    sim = similarity_from_embeddings(model_embs, gt_embs)
    return reward_from_similarity(model_items, gt_items, sim, **kwargs)

def summarize_result(name, result):
    return {
        "case": name,
        "reward": result["reward"],
        "semantic_fbeta": result["semantic_fbeta"],
        "precision": result["precision"],
        "recall": result["recall"],
        "avg_pair_penalty": result["avg_pair_penalty"],
        "matched_pairs": len(result["pairs"]),
        "unmatched_model": len(result["unmatched_model"]),
        "unmatched_ground_truth": len(result["unmatched_ground_truth"]),
    }

def show_result(result):
    summary = pd.DataFrame([{
        "reward": result["reward"],
        "semantic_fbeta": result["semantic_fbeta"],
        "precision": result["precision"],
        "recall": result["recall"],
        "avg_pair_penalty": result["avg_pair_penalty"],
        "unmatched_model": len(result["unmatched_model"]),
        "unmatched_ground_truth": len(result["unmatched_ground_truth"]),
    }])
    display(summary)
    if not result["pairs_df"].empty:
        display(result["pairs_df"])
    else:
        print("No matched pairs above threshold.")
    if result["unmatched_model"]:
        print("Unmatched model items:")
        for x in result["unmatched_model"]:
            print("  -", x)
    if result["unmatched_ground_truth"]:
        print("Unmatched ground-truth items:")
        for x in result["unmatched_ground_truth"]:
            print("  -", x)

## Quick sanity checks

These examples are deliberately simple. They show the behavior you care about:

- exact or near-exact agreement
- negation flip
- laterality flip
- dose mismatch
- extra generated content
- missing content
- optional protected-term mismatch for drug swaps


In [5]:
cases = {
    "perfect_match": (
        ["No chest pain.", "Metformin 500 mg PO daily."],
        ["No chest pain.", "Metformin 500 mg by mouth once daily."],
        {},
    ),
    "negation_flip": (
        ["Chest pain present."],
        ["No chest pain."],
        {},
    ),
    "laterality_flip": (
        ["Left lower extremity edema."],
        ["Right lower extremity edema."],
        {},
    ),
    "dose_flip": (
        ["Metformin 500 mg PO daily."],
        ["Metformin 1 g PO daily."],
        {},
    ),
    "extra_sentence": (
        ["No chest pain.", "Metformin 500 mg PO daily.", "Fever today."],
        ["No chest pain.", "Metformin 500 mg PO daily."],
        {},
    ),
    "missing_sentence": (
        ["No chest pain."],
        ["No chest pain.", "Metformin 500 mg PO daily."],
        {},
    ),
    "drug_swap_with_protected_terms": (
        ["Lisinopril 10 mg PO daily."],
        ["Losartan 10 mg PO daily."],
        {"protected_terms": {"lisinopril", "losartan"}},
    ),
}

results = {}
summary_rows = []

for name, (model_items, gt_items, extra_kwargs) in cases.items():
    result = reward_score_lexical(
        model_items,
        gt_items,
        sim_threshold=0.25,   # slightly lower for the lexical fallback
        **extra_kwargs,
    )
    results[name] = result
    summary_rows.append(summarize_result(name, result))

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["reward", "avg_pair_penalty"],
    ascending=[False, True],
).reset_index(drop=True)

summary_df

,case,reward,semantic_fbeta,precision,recall,avg_pair_penalty,matched_pairs,unmatched_model,unmatched_ground_truth
0,perfect_match,0.8548,0.8548,0.8548,0.8548,0.0,2,0,0
1,extra_sentence,0.8000,0.8000,0.6667,1.0000,0.0,2,1,0
2,missing_sentence,0.6667,0.6667,1.0000,0.5000,0.0,1,0,1
3,negation_flip,0.0000,0.3721,0.3721,0.3721,1.0,1,0,0
4,laterality_flip,0.0000,0.7115,0.7115,0.7115,1.0,1,0,0
5,drug_swap_with_protected_terms,0.0000,0.3373,0.3373,0.3373,1.0,1,0,0
6,dose_flip,0.0000,0.8014,0.8014,0.8014,1.2,1,0,0


In [6]:
show_result(results["negation_flip"])

,reward,semantic_fbeta,precision,recall,avg_pair_penalty,unmatched_model,unmatched_ground_truth
0,0.0,0.3721,0.3721,0.3721,1.0,0,0


,model_index,gt_index,similarity,model_text,gt_text,penalties,pair_penalty
0,0,0,0.3721,Chest pain present.,No chest pain.,negation:1.0,1.0


In [7]:
show_result(results["drug_swap_with_protected_terms"])

,reward,semantic_fbeta,precision,recall,avg_pair_penalty,unmatched_model,unmatched_ground_truth
0,0.0,0.3373,0.3373,0.3373,1.0,0,0


,model_index,gt_index,similarity,model_text,gt_text,penalties,pair_penalty
0,0,0,0.3373,Lisinopril 10 mg PO daily.,Losartan 10 mg PO daily.,protected_terms:1.0,1.0


## Try your own example

Enter either a list of strings or a single newline-separated string.


In [8]:
ground_truth = [
    "No chest pain.",
    "Metformin 500 mg PO daily.",
    "Family history of diabetes.",
]

model_output = [
    "Chest pain denied.",
    "Metformin 500 mg by mouth daily.",
    "Mother has diabetes.",
]

result = reward_score_lexical(
    model_output,
    ground_truth,
    sim_threshold=0.25,
    protected_terms={"metformin", "diabetes"},
)

show_result(result)

,reward,semantic_fbeta,precision,recall,avg_pair_penalty,unmatched_model,unmatched_ground_truth
0,0.541,0.541,0.541,0.541,0.0,0,0


,model_index,gt_index,similarity,model_text,gt_text,penalties,pair_penalty
0,0,0,0.4616,Chest pain denied.,No chest pain.,,0.0
1,1,1,0.7912,Metformin 500 mg by mouth daily.,Metformin 500 mg PO daily.,,0.0
2,2,2,0.3702,Mother has diabetes.,Family history of diabetes.,,0.0


## Swap in a real medical embedding model

For actual RL training, keep the reward logic the same and replace the lexical similarity matrix with normalized embedding dot products.

The flow you described is exactly what this notebook supports:

1. precompute and store **ground-truth embeddings**
2. encode **model outputs** for each rollout
3. compute `sim_matrix = model_embs @ gt_embs.T`
4. call `reward_score_from_embeddings(...)`

The cell below is intentionally commented out so the notebook stays runnable without extra packages.


In [9]:
# Optional:
# !pip install -U sentence-transformers

# from sentence_transformers import SentenceTransformer
#
# model_name = "ncbi/MedCPT-Query-Encoder"   # or another medical embedding model
# embedder = SentenceTransformer(model_name, device="cuda")
#
# gt_items = ["No chest pain.", "Metformin 500 mg PO daily."]
# model_items = ["Chest pain denied.", "Metformin 500 mg by mouth daily."]
#
# # Precompute once for the ground truth side
# gt_embs = embedder.encode(
#     gt_items,
#     normalize_embeddings=True,
#     batch_size=256,
#     convert_to_numpy=True,
# )
#
# # Compute on the fly for each rollout
# model_embs = embedder.encode(
#     model_items,
#     normalize_embeddings=True,
#     batch_size=256,
#     convert_to_numpy=True,
# )
#
# result = reward_score_from_embeddings(
#     model_items,
#     gt_items,
#     model_embs=model_embs,
#     gt_embs=gt_embs,
#     sim_threshold=0.35,
#     protected_terms={"metformin"},
# )
#
# show_result(result)

## Notes

A few practical points before you wire this into RL:

- The lexical backend is only a **playground backend**. Use a real medical encoder for training.
- `protected_terms` is a cheap hook for medication names, diagnoses, procedures, anatomy, or any tokens you want to treat as high-value.
- The default penalties are intentionally conservative. Tune them to your risk tolerance.
- For production, you will probably want a stronger concept layer later, but this notebook is a good first reward function to stress-test.
